# Notebook 01 - Data Collection

This notebook loads all raw data sources needed to build the Cybersecurity Readiness Index (CRI):

- **NCSI** (National Cyber Security Index) - scores for cybersecurity capacity, legal measures, and organisational measures
- **ITU GCI 2024** - Global Cybersecurity Index scores used for comparison
- **World Bank** - internet penetration (%), broadband subscriptions (per 100 people), GDP per capita (USD)
- **UN HDI** - Human Development Index used in the regression comparison notebook

All datasets are merged on country name and saved to `data/processed/merged_data.csv`.

**Data files expected in `data/raw/`:**
- `ncsi.csv`
- `itu_gci_2024.csv`
- `worldbank_internet.csv`
- `worldbank_broadband.csv`
- `worldbank_gdp.csv`
- `hdi_2023.csv`

In [ ]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

RAW = '../data/raw/'
PROCESSED = '../data/processed/'
os.makedirs(PROCESSED, exist_ok=True)

## 1. Load NCSI Data

The NCSI CSV is downloaded from https://ncsi.ega.ee. It includes scores for cybersecurity capacity, legal measures, and organisational readiness across ~160 countries. We select the columns relevant to our four sub-indices.

In [ ]:
ncsi_raw = pd.read_csv(f'{RAW}ncsi.csv')
print(f'NCSI raw shape: {ncsi_raw.shape}')
print(f'NCSI columns: {ncsi_raw.columns.tolist()}')
display(ncsi_raw.head())

In [ ]:
# Rename to standardised column names.
# Adjust these mappings if the NCSI CSV uses different column names.
ncsi_col_map = {
    'Country': 'country',
    'NCSI Score': 'ncsi_score',
    'Cybersecurity Policy Development': 'ncsi_policy',
    'Cyber Threat Analysis and Information': 'ncsi_threat_analysis',
    'Education and Professional Development': 'ncsi_education',
    'Contribution to Global Cybersecurity': 'ncsi_global',
    'Protection of Digital Services': 'ncsi_digital_services',
    'Protection of Essential Services': 'ncsi_essential_services',
    'E-identification and Trust Services': 'ncsi_eid',
    'Protection of Personal Data': 'ncsi_data_protection',
    'Cybercrime and e-Evidence': 'ncsi_cybercrime',
    'Military Cyber Operations': 'ncsi_military',
}

# Only keep columns that exist in this CSV
existing_cols = {k: v for k, v in ncsi_col_map.items() if k in ncsi_raw.columns}
ncsi = ncsi_raw[list(existing_cols.keys())].rename(columns=existing_cols)
ncsi['country'] = ncsi['country'].str.strip()
print(f'NCSI selected shape: {ncsi.shape}')
display(ncsi.head())

## 2. Load ITU GCI 2024 Data

The ITU Global Cybersecurity Index 2024 provides an overall score (0–100) and pillar scores. This is used both as a variable in our index and as a comparison benchmark in notebook 06.

In [ ]:
gci_raw = pd.read_csv(f'{RAW}itu_gci_2024.csv')
print(f'GCI raw shape: {gci_raw.shape}')
print(f'GCI columns: {gci_raw.columns.tolist()}')
display(gci_raw.head())

In [ ]:
# Adjust column names to match what is in your downloaded CSV
gci_col_map = {
    'Country': 'country',
    'GCI Score': 'gci_score',
    'Legal Measures': 'gci_legal',
    'Technical Measures': 'gci_technical',
    'Organizational Measures': 'gci_organisational',
    'Capacity Development': 'gci_capacity',
    'Cooperation': 'gci_cooperation',
}

existing_cols = {k: v for k, v in gci_col_map.items() if k in gci_raw.columns}
gci = gci_raw[list(existing_cols.keys())].rename(columns=existing_cols)
gci['country'] = gci['country'].str.strip()
print(f'GCI selected shape: {gci.shape}')
display(gci.head())

## 3. Load World Bank Data

World Bank indicators downloaded from https://data.worldbank.org:
- **IT.NET.USER.ZS** — Individuals using the Internet (% of population)
- **IT.NET.BBD.ZS** — Fixed broadband subscriptions (per 100 people)
- **NY.GDP.PCAP.CD** — GDP per capita (current USD)

World Bank CSVs have metadata rows at the top; we skip these and extract the most recent year available.

In [ ]:
def load_worldbank(filename, value_col_name, year_range=range(2018, 2024)):
    """Load a World Bank CSV, extract the most recent non-null year."""
    df = pd.read_csv(f'{RAW}{filename}', skiprows=4)
    # Keep country name and year columns
    year_cols = [str(y) for y in year_range if str(y) in df.columns]
    df = df[['Country Name'] + year_cols].rename(columns={'Country Name': 'country'})
    df['country'] = df['country'].str.strip()
    # Take the most recent non-null value across available years
    df[value_col_name] = df[year_cols].apply(
        lambda row: row.dropna().iloc[-1] if not row.dropna().empty else np.nan, axis=1
    )
    return df[['country', value_col_name]]


wb_internet = load_worldbank('worldbank_internet.csv', 'internet_penetration')
wb_broadband = load_worldbank('worldbank_broadband.csv', 'broadband_per_100')
wb_gdp = load_worldbank('worldbank_gdp.csv', 'gdp_per_capita')

print(f'Internet penetration: {wb_internet.shape}')
print(f'Broadband: {wb_broadband.shape}')
print(f'GDP per capita: {wb_gdp.shape}')
display(wb_internet.head())

## 4. Load HDI Data

The UN Human Development Index (HDI) is used in notebook 06 for regression comparison. Downloaded from https://hdr.undp.org/data-center/human-development-index.

In [ ]:
hdi_raw = pd.read_csv(f'{RAW}hdi_2023.csv')
print(f'HDI raw shape: {hdi_raw.shape}')
print(f'HDI columns: {hdi_raw.columns.tolist()}')
display(hdi_raw.head())

In [ ]:
# Adjust column names to match your downloaded CSV
hdi_col_map = {
    'Country': 'country',
    'Human Development Index (HDI)': 'hdi_score',
}
existing_cols = {k: v for k, v in hdi_col_map.items() if k in hdi_raw.columns}
hdi = hdi_raw[list(existing_cols.keys())].rename(columns=existing_cols)
hdi['country'] = hdi['country'].str.strip()
print(f'HDI selected shape: {hdi.shape}')
display(hdi.head())

## 5. Merge All Datasets

All datasets are merged on `country` using left joins anchored to the NCSI dataset (which defines our country list). Countries with no NCSI record are dropped. We keep 30–40 countries that appear in the most sources.

In [ ]:
merged = ncsi.copy()

for df, label in [(gci, 'GCI'), (wb_internet, 'Internet'), (wb_broadband, 'Broadband'),
                  (wb_gdp, 'GDP'), (hdi, 'HDI')]:
    before = len(merged)
    merged = merged.merge(df, on='country', how='left')
    print(f'After merging {label}: {len(merged)} rows (was {before})')

print(f'\nFinal merged shape: {merged.shape}')
display(merged.head())

## 6. Filter to Target Countries

We restrict the dataset to 35 countries that have broad data availability across all sources. These countries represent a mix of high-, middle-, and low-income nations to ensure the index is meaningful across the full spectrum.

In [ ]:
TARGET_COUNTRIES = [
    'United States', 'United Kingdom', 'Germany', 'France', 'Estonia',
    'Finland', 'Norway', 'Sweden', 'Denmark', 'Netherlands',
    'Australia', 'Canada', 'Japan', 'South Korea', 'Singapore',
    'Israel', 'Switzerland', 'Austria', 'Belgium', 'Spain',
    'Italy', 'Portugal', 'Poland', 'Czech Republic', 'Hungary',
    'Brazil', 'India', 'China', 'South Africa', 'Nigeria',
    'Kenya', 'Ghana', 'Mexico', 'Argentina', 'Turkey',
]

filtered = merged[merged['country'].isin(TARGET_COUNTRIES)].copy()
print(f'Countries matched: {len(filtered)}')

# Report any target countries not found
missing_countries = set(TARGET_COUNTRIES) - set(filtered['country'])
if missing_countries:
    print(f'Countries NOT found in NCSI data: {sorted(missing_countries)}')
    print('Consider checking country name spelling in the NCSI CSV.')

display(filtered)

## 7. Summary Statistics and Data Quality Check

In [ ]:
print('=== Data Types ===')
print(filtered.dtypes)
print()
print('=== Missing Values ===')
print(filtered.isnull().sum())
print()
print('=== Descriptive Statistics ===')
display(filtered.describe())

## 8. Save Merged Data

In [ ]:
output_path = f'{PROCESSED}merged_data.csv'
filtered.to_csv(output_path, index=False)
print(f'Saved merged data to {output_path}')
print(f'Shape: {filtered.shape}')